# Lesson 8: SpanMarker - State-of-the-Art Span-Based NER

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand span-based NER vs token-based approaches
2. Use pre-trained SpanMarker models for inference
3. Train custom SpanMarker models
4. Integrate SpanMarker with spaCy
5. Achieve state-of-the-art results on NER benchmarks

---

## 📚 Table of Contents

1. [Introduction to SpanMarker](#1-introduction-to-spanmarker)
2. [Span-Based vs Token-Based NER](#2-span-based-vs-token-based-ner)
3. [Using Pre-trained SpanMarker Models](#3-using-pre-trained-spanmarker-models)
4. [Training Custom SpanMarker Models](#4-training-custom-spanmarker-models)
5. [spaCy Integration](#5-spacy-integration)
6. [Comparison with Other Methods](#6-comparison-with-other-methods)
7. [Further Reading](#7-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q span_marker transformers datasets spacy

In [ ]:
# Import libraries
import torch
from span_marker import SpanMarkerModel
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

---

## 1. Introduction to SpanMarker

### What is SpanMarker?

> **SpanMarker** is a framework for training powerful Named Entity Recognition models using familiar encoders such as BERT, RoBERTa and DeBERTa. Built on top of 🤗 Transformers, it achieves state-of-the-art results.
>
> — [SpanMarker GitHub](https://github.com/tomaarsen/SpanMarkerNER)

### Key Features

| Feature | Description |
|---------|-------------|
| **State-of-the-art** | 93.1 F1 on CoNLL03 |
| **Span-based** | Predicts entity spans directly |
| **Flexible encoders** | BERT, RoBERTa, DeBERTa, multilingual |
| **Easy to use** | Familiar Transformers API |
| **spaCy integration** | Direct pipeline integration |
| **Hub support** | Many pre-trained models available |

### Performance Highlights

| Model | Dataset | F1 Score |
|-------|---------|----------|
| SpanMarker (XLM-RoBERTa-large) | CoNLL03 | **93.1** |
| SpanMarker (BERT-base) | FewNERD | **70.5** |
| SpanMarker (mBERT) | MultiNERD | **91.2** |

---

## 2. Span-Based vs Token-Based NER

### Token-Based Approach (BERT NER)

```
Input:  [CLS] Barack Obama visited Paris [SEP]
                 ↓      ↓       ↓      ↓
Output:        B-PER  I-PER    O    B-LOC

Problem: Must decode BIO tags → entities (error-prone)
```

### Span-Based Approach (SpanMarker)

```
Input:  "Barack Obama visited Paris"
                 ↓
Consider all spans:
  - (0,1): "Barack"        → Score for each entity type
  - (0,2): "Barack Obama"  → Score for each entity type  ✓ PER
  - (1,2): "Obama"         → Score for each entity type
  - (2,3): "visited"       → Score for each entity type
  - (3,4): "Paris"         → Score for each entity type  ✓ LOC
  ...

Output: [("Barack Obama", PER), ("Paris", LOC)]
```

### Advantages of Span-Based

| Aspect | Token-Based | Span-Based |
|--------|-------------|------------|
| Nested entities | Difficult | Native support |
| Entity boundaries | Implicit (BIO) | Explicit |
| Training signal | Per-token | Per-entity |
| Decoding | Required | Not needed |

In [ ]:
# Visualize span enumeration
def enumerate_spans(tokens, max_length=None):
    """Enumerate all possible spans in a sentence."""
    spans = []
    n = len(tokens)
    max_len = max_length or n
    
    for start in range(n):
        for end in range(start + 1, min(start + max_len + 1, n + 1)):
            span_text = ' '.join(tokens[start:end])
            spans.append((start, end, span_text))
    
    return spans

# Example
tokens = ["Barack", "Obama", "visited", "Paris"]
spans = enumerate_spans(tokens, max_length=3)

print("📊 All Possible Spans (max_length=3):\n")
print(f"{'Start':<8} {'End':<8} {'Span Text'}")
print("=" * 40)
for start, end, text in spans:
    print(f"{start:<8} {end:<8} {text}")

print(f"\n📈 Total spans: {len(spans)}")
print(f"   (Much fewer than all token combinations!)")

---

## 3. Using Pre-trained SpanMarker Models

### Loading Models

In [ ]:
# Load a pre-trained SpanMarker model
model = SpanMarkerModel.from_pretrained(
    "tomaarsen/span-marker-bert-base-fewnerd-fine-super"
)

print("✅ SpanMarker model loaded!")
print(f"   Labels: {model.config.id2label}")

In [ ]:
# Basic inference
text = "Amelia Earhart flew her single engine Lockheed Vega 5B across the Atlantic to Paris."

entities = model.predict(text)

print("🏷️ SpanMarker Predictions:\n")
print(f"Text: {text}\n")

for entity in entities:
    print(f"   • '{entity['span']}' → {entity['label']} (score: {entity['score']:.4f})")

In [ ]:
# Batch prediction
texts = [
    "Apple CEO Tim Cook announced the iPhone 15 at Apple Park.",
    "The European Union held meetings in Brussels, Belgium.",
    "Dr. Sarah Johnson works at Massachusetts General Hospital.",
    "Tesla's Elon Musk visited the Gigafactory in Austin, Texas."
]

print("⚡ Batch Prediction Results:\n")

batch_results = model.predict(texts)

for text, entities in zip(texts, batch_results):
    print(f"Text: {text}")
    if entities:
        for ent in entities:
            print(f"   • '{ent['span']}' → {ent['label']}")
    else:
        print("   • No entities found")
    print()

In [ ]:
# Understanding SpanMarker output format
print("📋 Entity Output Format:\n")

sample_entities = model.predict("Bill Gates founded Microsoft in Albuquerque.")

for i, entity in enumerate(sample_entities):
    print(f"Entity {i + 1}:")
    for key, value in entity.items():
        print(f"   {key}: {value}")
    print()

### Available Pre-trained Models

| Model | Dataset | F1 | Description |
|-------|---------|-----|-------------|
| `span-marker-bert-base-fewnerd-fine-super` | FewNERD | 70.5 | Fine-grained entity types |
| `span-marker-roberta-large-fewnerd-fine-super` | FewNERD | 71.5 | Larger, more accurate |
| `span-marker-xlm-roberta-large-conll03` | CoNLL03 | 93.1 | SOTA on CoNLL03 |
| `span-marker-mbert-base-multinerd` | MultiNERD | 91.2 | Multilingual |

> **Browse all models**: [Hugging Face SpanMarker Models](https://huggingface.co/models?library=span-marker)

In [ ]:
# Try a different model (CoNLL03 for standard NER)
model_conll = SpanMarkerModel.from_pretrained(
    "tomaarsen/span-marker-bert-base-conll03"
)

test_text = "Barack Obama met Angela Merkel in Berlin to discuss NATO policies."

entities_conll = model_conll.predict(test_text)

print("🏷️ CoNLL03 Model (PER, ORG, LOC, MISC):\n")
print(f"Text: {test_text}\n")

for entity in entities_conll:
    print(f"   • '{entity['span']}' → {entity['label']} ({entity['score']:.3f})")

---

## 4. Training Custom SpanMarker Models

### Dataset Requirements

SpanMarker accepts datasets with:
- `tokens`: List of tokens
- `ner_tags`: List of NER tags (IOB, IOB2, BIOES, or plain labels)

In [ ]:
# Load a dataset for training
from datasets import load_dataset

dataset = load_dataset("conll2003", trust_remote_code=True)

# Get label names
label_names = dataset["train"].features["ner_tags"].feature.names

print("📊 CoNLL2003 Dataset:")
print(f"   Train: {len(dataset['train'])} examples")
print(f"   Validation: {len(dataset['validation'])} examples")
print(f"   Test: {len(dataset['test'])} examples")
print(f"   Labels: {label_names}")

In [ ]:
# Initialize a SpanMarker model for training
from span_marker import SpanMarkerModel, Trainer, SpanMarkerModelCardData
from transformers import TrainingArguments

# Use a smaller dataset for demo
train_dataset = dataset["train"].select(range(1000))
eval_dataset = dataset["validation"].select(range(200))

# Initialize model from encoder
model_custom = SpanMarkerModel.from_pretrained(
    "bert-base-cased",  # Base encoder
    labels=label_names,
    model_max_length=256,
    marker_max_length=128,
    entity_max_length=8,  # Max entity length in words
)

print("✅ SpanMarker model initialized for training")
print(f"   Encoder: bert-base-cased")
print(f"   Labels: {label_names}")

In [ ]:
# Set up training arguments
args = TrainingArguments(
    output_dir="./spanmarker-custom",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,  # Use more epochs for real training
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# Create trainer
trainer = Trainer(
    model=model_custom,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("✅ Trainer configured")

In [ ]:
# Train the model
print("🏋️ Training SpanMarker model...\n")

trainer.train()

print("\n✅ Training complete!")

In [ ]:
# Evaluate
print("📊 Evaluation Results:\n")

metrics = trainer.evaluate()

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# Test trained model
test_text = "Google announced new AI features at their I/O conference in Mountain View."

print("🧪 Testing Trained Model:\n")
print(f"Text: {test_text}\n")

entities_trained = model_custom.predict(test_text)

for entity in entities_trained:
    print(f"   • '{entity['span']}' → {entity['label']} ({entity['score']:.3f})")

---

## 5. spaCy Integration

SpanMarker models can be easily integrated into spaCy pipelines.

In [ ]:
# Install spacy-span-marker integration
!pip install -q spacy-span-marker

In [ ]:
import spacy

# Load spaCy with SpanMarker
nlp = spacy.blank("en")

# Add SpanMarker to the pipeline
nlp.add_pipe(
    "span_marker",
    config={
        "model": "tomaarsen/span-marker-bert-base-fewnerd-fine-super",
        "batch_size": 4,
        "device": device,
    }
)

print("✅ SpanMarker added to spaCy pipeline")
print(f"   Pipeline: {nlp.pipe_names}")

In [ ]:
# Use spaCy with SpanMarker
text = "Bill Gates and Steve Jobs revolutionized the tech industry. Microsoft and Apple became competitors."

doc = nlp(text)

print("🏷️ spaCy + SpanMarker Results:\n")
print(f"Text: {text}\n")

# Entities are stored in doc.ents
for ent in doc.ents:
    print(f"   • '{ent.text}' → {ent.label_}")

In [ ]:
# Use displaCy for visualization
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

---

## 6. Comparison with Other Methods

In [ ]:
# Compare SpanMarker with BERT NER and spaCy
from transformers import pipeline
import time

# Load models
bert_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
spacy_nlp = spacy.load("en_core_web_sm")
spanmarker_model = SpanMarkerModel.from_pretrained("tomaarsen/span-marker-bert-base-conll03")

test_texts = [
    "Apple CEO Tim Cook announced the iPhone 15 in Cupertino.",
    "The European Union met in Brussels to discuss NATO policies.",
    "Dr. Fauci spoke at the National Institutes of Health about COVID-19."
]

print("📊 Model Comparison\n")
print("=" * 80)

for text in test_texts:
    print(f"\nText: {text}\n")
    
    # BERT NER
    start = time.time()
    bert_results = bert_ner(text)
    bert_time = (time.time() - start) * 1000
    bert_entities = [(r['word'], r['entity_group']) for r in bert_results]
    
    # spaCy
    start = time.time()
    doc = spacy_nlp(text)
    spacy_time = (time.time() - start) * 1000
    spacy_entities = [(ent.text, ent.label_) for ent in doc.ents]
    
    # SpanMarker
    start = time.time()
    sm_results = spanmarker_model.predict(text)
    sm_time = (time.time() - start) * 1000
    sm_entities = [(r['span'], r['label']) for r in sm_results]
    
    print(f"   BERT NER ({bert_time:.1f}ms): {bert_entities}")
    print(f"   spaCy ({spacy_time:.1f}ms):    {spacy_entities}")
    print(f"   SpanMarker ({sm_time:.1f}ms): {sm_entities}")

### When to Use SpanMarker

| Use Case | Recommendation |
|----------|---------------|
| Maximum accuracy needed | ✅ SpanMarker |
| Nested entities | ✅ SpanMarker |
| Fine-grained entity types | ✅ SpanMarker (FewNERD) |
| Speed critical | ❌ Use spaCy |
| Zero-shot custom entities | ❌ Use GLiNER |
| Simple integration | ✅ SpanMarker (spaCy) |

---

## 7. Further Reading

### 📚 Resources

- [SpanMarker GitHub](https://github.com/tomaarsen/SpanMarkerNER)
- [SpanMarker Documentation](https://tomaarsen.github.io/SpanMarkerNER)
- [Hugging Face Models](https://huggingface.co/models?library=span-marker)
- [Using SpanMarker at Hugging Face](https://huggingface.co/docs/hub/en/span_marker)

### 📖 Related Papers

- Span-based NER approaches
- FewNERD: A Few-shot Named Entity Recognition Dataset

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **Span-based vs Token-based**: Direct span prediction vs BIO decoding
2. **Pre-trained models**: Using existing SpanMarker models
3. **Custom training**: Training on your own datasets
4. **spaCy integration**: Seamless pipeline integration
5. **Comparisons**: When to use SpanMarker vs other methods

In [ ]:
# Cleanup
import shutil
import os

if os.path.exists("./spanmarker-custom"):
    shutil.rmtree("./spanmarker-custom")

print("🎉 Congratulations! You've completed Lesson 8: SpanMarker NER")
print("\n📝 Key takeaways:")
print("   1. SpanMarker predicts entity spans directly")
print("   2. Achieves SOTA 93.1 F1 on CoNLL03")
print("   3. Supports nested entities naturally")
print("   4. Easy spaCy integration available")
print("\n👉 Continue to Lesson 9: LLM-based NER")